# Case Study 2 — NeoBERT Phase 1 LoRA Fine-Tuning for C/C++ Vulnerability Detection

This notebook starts **Case Study 2** using the same binary vulnerability-detection task as Case Study 1:

```text
C/C++ function  →  label ∈ {0,1}
0 = safe / non-vulnerable
1 = vulnerable
```

## Why this notebook exists

Case Study 1 used classical and shallow ML pipelines such as TF-IDF, SVD, static features, Logistic Regression, Random Forest, and MLP.

Case Study 2 uses a deep contextual encoder:

```text
Input C/C++ function
→ NeoBERT-250M encoder
→ Phase 1: LoRA parameter-efficient fine-tuning
→ sequence classification head
→ vulnerability prediction + confidence
```

## Important methodology guard

This notebook is **development-only**. It does **not** evaluate on the frozen 20% final holdout.

The frozen outer holdout has already been used for the selected Case Study 1 final model. For Case Study 2 development, we use the existing project-disjoint inner fold manifest.

## Recommended run order

1. Run a small smoke test first.
2. Run one full project-disjoint development fold.
3. Only after the LoRA pipeline is stable, build Phase 2 ReFT / HEFT.

## 0. Research-based design choices

NeoBERT is loaded from `chandar-lab/NeoBERT` using Hugging Face `trust_remote_code=True`.

Key design choices used here:

- **Backbone**: NeoBERT-250M.
- **Context cap**: up to 4096 tokens, matching the project slide.
- **Task**: binary sequence classification.
- **Fine-tuning style**: LoRA on the NeoBERT attention projection module.
- **Class imbalance**: weighted cross-entropy based on training-fold class frequencies.
- **Evaluation**: project-disjoint validation fold, PR-AUC as the primary metric.

Implementation note: NeoBERT's current implementation uses a fused attention projection layer named `qkv`, rather than separate `W_q`, `W_k`, and `W_v` modules. Therefore, the LoRA target is `qkv`, which covers the fused query/key/value projection.

### PATCHED V7 memory-safe update

This version defaults to a Colab T4/L4-safe context length (`MAX_LENGTH=1024`) because NeoBERT attention memory grows quadratically with sequence length. The slide-level target context is 4096 tokens, but 4096-token LoRA training with the current remote-code SDPA path can exceed 14–16 GB GPUs. Use 2048/4096 only on larger GPUs after the 1024-token run succeeds.


In [1]:
# ============================================================
# 1. Install dependencies
# ============================================================
# Safe Colab install cell.
# Set CUDA allocator behavior before importing torch to reduce fragmentation after long batches.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Important: do NOT install or downgrade torch manually.
# Colab GPU runtimes already include a compatible PyTorch build.
#
# Why uninstall torchao?
# Some Colab environments contain an older torchao package. Recent PEFT
# checks torchao if it is installed, and older torchao versions can make
# get_peft_model(...) fail even though we are not using torchao/quantization.
# Removing torchao makes PEFT skip that optional integration path.

INSTALL_PACKAGES = True
REMOVE_OPTIONAL_TORCHAO = True

if INSTALL_PACKAGES:
    %pip -q install -U \
        "transformers>=4.48.2" \
        "accelerate>=1.0.0" \
        "datasets>=2.20.0" \
        "peft>=0.13.0" \
        "scikit-learn>=1.3.0" \
        "pyarrow>=14.0.0" \
        "sentencepiece" \
        "evaluate" \
        "tqdm"

if REMOVE_OPTIONAL_TORCHAO:
    import sys, subprocess, importlib.util
    if importlib.util.find_spec("torchao") is not None:
        print("Removing optional torchao package to avoid PEFT compatibility conflicts...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
    else:
        print("torchao is not installed; no removal needed.")

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Use Runtime → Change runtime type → GPU.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 44.3 MB/s eta 0:00:00
Removing optional torchao package to avoid PEFT compatibility conflicts...
Torch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Note on `torchao`, PEFT, and mixed precision

This notebook removes optional `torchao` when needed to avoid PEFT dispatch conflicts. It also disables `Trainer(fp16=True)` because NeoBERT is loaded in FP16 for memory, and PEFT/Accelerate can otherwise fail with `Attempting to unscale FP16 gradients`. The model still runs on GPU; this setting only disables the extra GradScaler path.


In [2]:
# ============================================================
# 2. Imports and environment
# ============================================================
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import json
import math
import time
import random
import inspect
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

from peft import LoraConfig, TaskType, get_peft_model

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

# ------------------------------------------------------------
# NeoBERT compatibility helper
# ------------------------------------------------------------
def install_xformers_swiglu_stub():
    """Provide a tiny xformers.ops.SwiGLU replacement when xformers is absent.

    Some NeoBERT remote-code revisions import `xformers.ops.SwiGLU`.
    Installing xformers in Colab can force large CUDA/PyTorch dependency changes.
    This shim satisfies the import and implements the same SwiGLU MLP shape used
    by NeoBERT: w12 projects to 2*hidden, split into gate/up, then w3 projects back.
    """
    try:
        import xformers  # noqa: F401
        from xformers.ops import SwiGLU  # noqa: F401
        print("xformers is available; no SwiGLU shim needed.")
        return
    except Exception:
        pass

    import sys
    import types
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class SwiGLU(nn.Module):
        def __init__(self, in_features, hidden_features, out_features, bias=False, **kwargs):
            super().__init__()
            self.w12 = nn.Linear(in_features, 2 * hidden_features, bias=bias)
            self.w3 = nn.Linear(hidden_features, out_features, bias=bias)

        def forward(self, x):
            w1, w2 = self.w12(x).chunk(2, dim=-1)
            return self.w3(F.silu(w1) * w2)

    xformers_mod = types.ModuleType("xformers")
    ops_mod = types.ModuleType("xformers.ops")
    ops_mod.SwiGLU = SwiGLU
    xformers_mod.ops = ops_mod
    sys.modules["xformers"] = xformers_mod
    sys.modules["xformers.ops"] = ops_mod
    print("Installed lightweight xformers.ops.SwiGLU compatibility shim.")


Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory GB: 14.56


In [3]:
# ============================================================
# 3. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# ============================================================
# 4. Global configuration
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Main Drive root used in Case Study 1.
DATA_ROOT = Path("/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData")

PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

EXPERIMENT_ID = "cs1_project_holdout20_innercv_v1"

DATA_PATH = PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
INNER_MANIFEST_PATH = MANIFEST_ROOT / EXPERIMENT_ID / "inner_cv" / "cs1_project_grouped_5fold_manifest.parquet"
OUTER_MANIFEST_PATH = MANIFEST_ROOT / EXPERIMENT_ID / "outer_holdout" / "cs1_outer_project_holdout_manifest.parquet"

# Use a new output directory because max_length changed from earlier attempted runs.
CS2_OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_ID / "case_study_2_neobert_lora_phase1_dev_v3_pilot_mode"
CS2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model and input.
MODEL_NAME = "chandar-lab/NeoBERT"
CODE_COLUMN = "normalized_code"
LABEL_COLUMN = "label"
PROJECT_COLUMN = "project"
ID_COLUMN = "source_row_id"
FOLD_COLUMN = "fold"

# ------------------------------------------------------------
# Memory policy
# ------------------------------------------------------------
# NeoBERT supports up to 4096 tokens, matching the intended design slide.
# However, LoRA training with 4096-token attention can exceed a 14–16 GB Colab GPU
# because attention memory scales approximately with sequence_length^2.
#
# Safe sequence:
#   1) Run smoke test with MAX_LENGTH=1024.
#   2) Run one full project-disjoint fold with MAX_LENGTH=1024.
#   3) Only try 2048/4096 on a larger GPU (A100/H100) after 1024 works.
MAX_LENGTH = 1024

# Run controls.
# Run controls. Exactly one of these should be True.
#
# Recommended sequence on Colab/T4-like GPU:
#   1) Smoke test: checks that tokenizer/model/LoRA/training/evaluation all work.
#   2) Pilot fold: larger project-disjoint experiment with capped rows.
#   3) Full fold: only if you have many hours of stable GPU time.
RUN_SMOKE_TEST = True
RUN_PILOT_FOLD = False
RUN_ONE_FULL_FOLD = False
RUN_FULL_5FOLD_CV = False   # intentionally disabled for now; expensive

# Which project-disjoint fold to use for the first LoRA development run.
# Fold 4 means "Fold 5/5" if displayed 1-indexed.
FOLD_TO_RUN = 4

# Smoke test row caps. Keep these small on a free Colab/T4-like GPU.
# The smoke split is intentionally enriched with positives so the tiny run has
# enough vulnerable examples to verify ranking/threshold code. It is NOT a reportable metric.
SMOKE_TRAIN_ROWS = 512
SMOKE_VALID_ROWS = 256

# Pilot row caps. This is the next meaningful run after smoke.
# Training is still enriched with positives for learning signal; validation is
# random/natural from the held-out projects, so its positive rate is closer to reality.
PILOT_TRAIN_ROWS = 8192
PILOT_VALID_ROWS = 4096

# Training parameters.
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

# Memory-safe micro-batching. Increase only after a successful run.
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 32

# Keeping checkpointing off by default avoids incompatibility with custom remote model code.
# If the model later supports it cleanly, it can be tested as an optimization only.
USE_GRADIENT_CHECKPOINTING = False

# LoRA hyperparameters.
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# NeoBERT uses a fused qkv attention projection.
LORA_TARGET_MODULES = ["qkv"]

print("Output directory:", CS2_OUTPUT_DIR)
print("Data path:", DATA_PATH)
print("Inner manifest:", INNER_MANIFEST_PATH)
print("Outer manifest:", OUTER_MANIFEST_PATH)
print("Model:", MODEL_NAME)
print("Code column:", CODE_COLUMN)
print("Max token length:", MAX_LENGTH)
print("Run smoke test:", RUN_SMOKE_TEST)
print("Run pilot fold:", RUN_PILOT_FOLD)
print("Run one full fold:", RUN_ONE_FULL_FOLD)
print("Pilot train rows:", PILOT_TRAIN_ROWS)
print("Pilot valid rows:", PILOT_VALID_ROWS)
print("Per-device train batch size:", PER_DEVICE_TRAIN_BATCH_SIZE)
print("Gradient accumulation steps:", GRADIENT_ACCUMULATION_STEPS)


Output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/case_study_2_neobert_lora_phase1_dev_v3_pilot_mode
Data path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet
Inner manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet
Outer manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet
Model: chandar-lab/NeoBERT
Code column: normalized_code
Max token length: 1024
Run smoke test: True
Run pilot fold: False
Run one full fold: False
Pilot train rows: 8192
Pilot valid rows: 4096
Per-device train batch size: 1
Gradient accumulation steps: 32


In [5]:
# ============================================================
# 5. Helper functions
# ============================================================

def now():
    return datetime.now().strftime("%H:%M:%S")


def log(message: str):
    print(f"[{now()}] {message}", flush=True)


def require_path(path: Path, name: str):
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    log(f"{name} exists: {path}")


def find_outer_split_column(df: pd.DataFrame):
    candidates = ["split", "outer_split", "partition", "set", "subset"]
    for col in candidates:
        if col in df.columns:
            return col
    bool_candidates = ["is_outer_holdout", "outer_holdout", "is_holdout", "is_test"]
    for col in bool_candidates:
        if col in df.columns:
            return col
    return None


def stratified_cap(df: pd.DataFrame, max_rows: int, label_col: str, seed: int) -> pd.DataFrame:
    """Return a small stratified subset for smoke testing."""
    if max_rows is None or len(df) <= max_rows:
        return df.copy()

    pos = df[df[label_col] == 1]
    neg = df[df[label_col] == 0]

    # Keep enough positives so the smoke-test metric is meaningful.
    target_pos = min(len(pos), max(50, int(max_rows * 0.25)))
    target_neg = max_rows - target_pos

    pos_s = pos.sample(n=target_pos, random_state=seed) if len(pos) > target_pos else pos
    neg_s = neg.sample(n=min(len(neg), target_neg), random_state=seed) if len(neg) > target_neg else neg

    out = pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out


def random_cap(df: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    """Return a random subset, preserving the natural label distribution in expectation."""
    if max_rows is None or len(df) <= max_rows:
        return df.copy().reset_index(drop=True)
    return df.sample(n=max_rows, random_state=seed).reset_index(drop=True)


def balanced_training_cap(df: pd.DataFrame, max_rows: int, label_col: str, seed: int, positive_fraction: float = 0.25) -> pd.DataFrame:
    """Return a capped training subset enriched with positives.

    This is useful for transformer pilot runs under limited compute. The validation
    split should remain natural/random for honest metrics.
    """
    if max_rows is None or len(df) <= max_rows:
        return df.copy().reset_index(drop=True)

    pos = df[df[label_col] == 1]
    neg = df[df[label_col] == 0]
    target_pos = min(len(pos), max(1, int(max_rows * positive_fraction)))
    target_neg = max_rows - target_pos

    pos_s = pos.sample(n=target_pos, random_state=seed) if len(pos) > target_pos else pos
    neg_s = neg.sample(n=min(len(neg), target_neg), random_state=seed + 17) if len(neg) > target_neg else neg
    return pd.concat([pos_s, neg_s], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)


def compute_binary_metrics_from_scores(y_true, y_score, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    out = {}
    out["n_samples"] = int(len(y_true))
    out["vulnerable_1"] = int((y_true == 1).sum())
    out["non_vulnerable_0"] = int((y_true == 0).sum())
    out["positive_rate"] = float((y_true == 1).mean()) if len(y_true) else float("nan")
    out["threshold"] = float(threshold)

    try:
        out["average_precision_pr_auc"] = float(average_precision_score(y_true, y_score))
    except Exception:
        out["average_precision_pr_auc"] = float("nan")

    try:
        out["roc_auc"] = float(roc_auc_score(y_true, y_score))
    except Exception:
        out["roc_auc"] = float("nan")

    out["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["recall"] = float(recall_score(y_true, y_pred, zero_division=0))
    out["f1"] = float(f1_score(y_true, y_pred, zero_division=0))

    try:
        out["mcc"] = float(matthews_corrcoef(y_true, y_pred))
    except Exception:
        out["mcc"] = float("nan")

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out["true_negative"] = int(tn)
    out["false_positive"] = int(fp)
    out["false_negative"] = int(fn)
    out["true_positive"] = int(tp)
    out["specificity"] = float(tn / (tn + fp)) if (tn + fp) else float("nan")
    out["false_positive_rate"] = float(fp / (tn + fp)) if (tn + fp) else float("nan")
    out["false_negative_rate"] = float(fn / (fn + tp)) if (fn + tp) else float("nan")
    out["predicted_positive"] = int(y_pred.sum())
    out["predicted_positive_rate"] = float(y_pred.mean()) if len(y_pred) else float("nan")
    return out


def select_best_f1_threshold(y_true, y_score):
    grid = np.round(np.arange(0.05, 0.951, 0.01), 3)
    best = {"threshold": 0.5, "f1": -1.0}
    for thr in grid:
        pred = (y_score >= thr).astype(int)
        val = f1_score(y_true, pred, zero_division=0)
        if val > best["f1"]:
            best = {"threshold": float(thr), "f1": float(val)}
    return best


def softmax_positive(logits):
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return probs[:, 1]


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

In [6]:
# ============================================================
# 6. Load processed data and manifests
# ============================================================

require_path(DATA_PATH, "Processed dataset")
require_path(INNER_MANIFEST_PATH, "Inner CV manifest")
if OUTER_MANIFEST_PATH.exists():
    log(f"Outer holdout manifest exists: {OUTER_MANIFEST_PATH}")
else:
    log("Outer holdout manifest not found. Continuing with inner manifest only.")

df = pd.read_parquet(DATA_PATH)
inner_manifest = pd.read_parquet(INNER_MANIFEST_PATH)

log(f"Dataset shape: {df.shape}")
log(f"Inner manifest shape: {inner_manifest.shape}")

required_data_cols = {ID_COLUMN, CODE_COLUMN, LABEL_COLUMN, PROJECT_COLUMN}
missing_data_cols = sorted(required_data_cols - set(df.columns))
if missing_data_cols:
    raise KeyError(f"Dataset is missing required columns: {missing_data_cols}. Available: {list(df.columns)}")

required_manifest_cols = {ID_COLUMN, FOLD_COLUMN}
missing_manifest_cols = sorted(required_manifest_cols - set(inner_manifest.columns))
if missing_manifest_cols:
    raise KeyError(f"Inner manifest is missing required columns: {missing_manifest_cols}. Available: {list(inner_manifest.columns)}")

# Join the inner fold assignment into the dataset.
dev_df = df.merge(inner_manifest[[ID_COLUMN, FOLD_COLUMN]], on=ID_COLUMN, how="inner")

# Basic cleanup.
dev_df[CODE_COLUMN] = dev_df[CODE_COLUMN].fillna("").astype(str)
empty_mask = dev_df[CODE_COLUMN].str.strip().eq("")
if empty_mask.any():
    raise ValueError(f"{int(empty_mask.sum())} development rows contain empty {CODE_COLUMN}.")

dev_df[LABEL_COLUMN] = dev_df[LABEL_COLUMN].astype(int)

log(f"Development frame shape after inner-manifest join: {dev_df.shape}")
log(f"Development positives: {int(dev_df[LABEL_COLUMN].sum())} / {len(dev_df)} = {dev_df[LABEL_COLUMN].mean():.4%}")
log(f"Unique development projects: {dev_df[PROJECT_COLUMN].nunique()}")
log(f"Folds found: {sorted(dev_df[FOLD_COLUMN].unique().tolist())}")

# Safety check with outer holdout if available.
if OUTER_MANIFEST_PATH.exists():
    outer_manifest = pd.read_parquet(OUTER_MANIFEST_PATH)
    split_col = find_outer_split_column(outer_manifest)
    log(f"Outer manifest columns: {list(outer_manifest.columns)}")
    if split_col is not None:
        log(f"Detected outer split column: {split_col}")
    else:
        log("Could not auto-detect outer split column. This is only a warning; inner CV is still used.")

[16:46:18] Processed dataset exists: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet
[16:46:18] Inner CV manifest exists: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet
[16:46:19] Outer holdout manifest exists: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet
[16:46:31] Dataset shape: (261667, 5)
[16:46:31] Inner manifest shape: (203958, 4)
[16:46:33] Development frame shape after inner-manifest join: (203958, 6)
[16:46:33] Development positives: 10727 / 203958 = 5.2594%
[16:46:33] Unique development projects: 594
[16:46:33] Folds found: [0, 1, 2, 3, 4]
[16:46:34] Outer manifest columns: ['source_row_id', 'label', 'project', 'partition', 'outer_holdout_fold']
[16:46:3

In [7]:
# ============================================================
# 7. Build one project-disjoint train/validation fold
# ============================================================

fold_values = sorted(dev_df[FOLD_COLUMN].unique().tolist())
if FOLD_TO_RUN not in fold_values:
    raise ValueError(f"FOLD_TO_RUN={FOLD_TO_RUN} not found. Available folds: {fold_values}")

train_df = dev_df[dev_df[FOLD_COLUMN] != FOLD_TO_RUN].copy()
valid_df = dev_df[dev_df[FOLD_COLUMN] == FOLD_TO_RUN].copy()

train_projects = set(train_df[PROJECT_COLUMN].astype(str))
valid_projects = set(valid_df[PROJECT_COLUMN].astype(str))
overlap_projects = train_projects & valid_projects

if overlap_projects:
    raise RuntimeError(f"Project leakage detected: {len(overlap_projects)} overlapping projects.")
else:
    log("Project-disjoint fold verified: no train/validation project overlap.")

log(f"Fold {FOLD_TO_RUN}: train rows={len(train_df):,}, valid rows={len(valid_df):,}")
log(f"Fold {FOLD_TO_RUN}: train projects={train_df[PROJECT_COLUMN].nunique():,}, valid projects={valid_df[PROJECT_COLUMN].nunique():,}")
log(f"Fold {FOLD_TO_RUN}: train positive rate={train_df[LABEL_COLUMN].mean():.4%}, valid positive rate={valid_df[LABEL_COLUMN].mean():.4%}")

selected_run_modes = [RUN_SMOKE_TEST, RUN_PILOT_FOLD, RUN_ONE_FULL_FOLD]
if sum(bool(x) for x in selected_run_modes) != 1:
    raise RuntimeError(
        "Choose exactly one run mode: RUN_SMOKE_TEST, RUN_PILOT_FOLD, or RUN_ONE_FULL_FOLD."
    )

if RUN_SMOKE_TEST:
    log("RUN_SMOKE_TEST=True: using tiny positive-enriched subsets only. Metrics are not reportable.")
    train_df = stratified_cap(train_df, SMOKE_TRAIN_ROWS, LABEL_COLUMN, SEED)
    valid_df = stratified_cap(valid_df, SMOKE_VALID_ROWS, LABEL_COLUMN, SEED + 1)
    log(f"Smoke train rows={len(train_df):,}, smoke valid rows={len(valid_df):,}")
    log(f"Smoke train positive rate={train_df[LABEL_COLUMN].mean():.4%}, smoke valid positive rate={valid_df[LABEL_COLUMN].mean():.4%}")

elif RUN_PILOT_FOLD:
    log("RUN_PILOT_FOLD=True: using capped project-disjoint pilot subsets.")
    log("Pilot training is positive-enriched; pilot validation is natural/random from held-out projects.")
    train_df = balanced_training_cap(train_df, PILOT_TRAIN_ROWS, LABEL_COLUMN, SEED, positive_fraction=0.25)
    valid_df = random_cap(valid_df, PILOT_VALID_ROWS, SEED + 1)
    log(f"Pilot train rows={len(train_df):,}, pilot valid rows={len(valid_df):,}")
    log(f"Pilot train positive rate={train_df[LABEL_COLUMN].mean():.4%}, pilot valid positive rate={valid_df[LABEL_COLUMN].mean():.4%}")

else:
    log("RUN_ONE_FULL_FOLD=True: using all rows for this project-disjoint development fold.")
    log("This can take many hours on Colab/T4-like GPUs.")

[16:46:34] Project-disjoint fold verified: no train/validation project overlap.
[16:46:34] Fold 4: train rows=170,388, valid rows=33,570
[16:46:34] Fold 4: train projects=442, valid projects=152
[16:46:34] Fold 4: train positive rate=5.0590%, valid positive rate=6.2764%
[16:46:34] RUN_SMOKE_TEST=True: using tiny positive-enriched subsets only. Metrics are not reportable.
[16:46:34] Smoke train rows=512, smoke valid rows=256
[16:46:34] Smoke train positive rate=25.0000%, smoke valid positive rate=25.0000%


In [8]:
# ============================================================
# 8. Load tokenizer and tokenize C/C++ functions
# ============================================================

# NeoBERT's config points to google-bert/bert-base-uncased as the tokenizer source.
# Loading the tokenizer from NeoBERT with trust_remote_code=True can unnecessarily
# import NeoBERT's modeling file and fail if xformers is absent. The tokenizer itself
# is standard BERT WordPiece, so load it directly from google-bert.
TOKENIZER_NAME = "google-bert/bert-base-uncased"

log(f"Loading tokenizer: {TOKENIZER_NAME}")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.unk_token


def to_hf_dataset(frame: pd.DataFrame) -> Dataset:
    """Build an HF dataset for model training.

    We intentionally include only code and label here. Metadata such as
    source_row_id/project is kept in the external pandas DataFrames
    (`train_df`, `valid_df`) for artifact saving. String metadata must not be
    sent into DataCollatorWithPadding because it tries to convert every field
    in the batch into tensors, which causes errors such as:

        ValueError: too many dimensions 'str' ... features (`project`)
    """
    small = frame[[CODE_COLUMN, LABEL_COLUMN]].copy()
    small = small.rename(columns={LABEL_COLUMN: "labels"})
    small["labels"] = small["labels"].astype(int)
    return Dataset.from_pandas(small, preserve_index=False)


train_ds_raw = to_hf_dataset(train_df)
valid_ds_raw = to_hf_dataset(valid_df)


def tokenize_batch(batch):
    return tokenizer(
        batch[CODE_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


log("Tokenizing train dataset...")
train_ds = train_ds_raw.map(tokenize_batch, batched=True, remove_columns=[CODE_COLUMN])
log("Tokenizing validation dataset...")
valid_ds = valid_ds_raw.map(tokenize_batch, batched=True, remove_columns=[CODE_COLUMN])

# Sanity check: the maximum tensorized sequence length after truncation must not exceed MAX_LENGTH.
# On 14–16 GB GPUs this check is important because one unexpected 4096-token batch can OOM.
_train_sample_lengths = [len(x) for x in train_ds["input_ids"][: min(1000, len(train_ds))]]
_valid_sample_lengths = [len(x) for x in valid_ds["input_ids"][: min(1000, len(valid_ds))]]
print("Train token length sample: mean=%.1f, p95=%.1f, max=%d, cap=%d" % (
    float(np.mean(_train_sample_lengths)),
    float(np.percentile(_train_sample_lengths, 95)),
    int(np.max(_train_sample_lengths)),
    int(MAX_LENGTH),
))
print("Validation token length sample: mean=%.1f, p95=%.1f, max=%d, cap=%d" % (
    float(np.mean(_valid_sample_lengths)),
    float(np.percentile(_valid_sample_lengths, 95)),
    int(np.max(_valid_sample_lengths)),
    int(MAX_LENGTH),
))
assert max(_train_sample_lengths + _valid_sample_lengths) <= MAX_LENGTH


# Trainer/DataCollator should receive only tensorizable model fields.
# Keep source_row_id/project outside the HF Dataset and use valid_df when saving predictions.
model_cols = ["labels", "input_ids", "attention_mask"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in model_cols])
valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in model_cols])

train_ds.set_format(type=None, columns=model_cols)
valid_ds.set_format(type=None, columns=model_cols)

lengths = [len(x) for x in valid_ds["input_ids"][: min(1000, len(valid_ds))]]
log(f"Validation token length sample: mean={np.mean(lengths):.1f}, max={np.max(lengths)}, cap={MAX_LENGTH}")
print("Train dataset columns:", train_ds.column_names)
print("Validation dataset columns:", valid_ds.column_names)


[16:46:35] Loading tokenizer: google-bert/bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[16:46:37] Tokenizing train dataset...


Map:   0%|          | 0/512 [00:00<?, ? examples/s]

[16:46:40] Tokenizing validation dataset...


Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Train token length sample: mean=319.1, p95=1024.0, max=1024, cap=1024
Validation token length sample: mean=389.5, p95=1024.0, max=1024, cap=1024
[16:46:41] Validation token length sample: mean=389.5, max=1024, cap=1024
Train dataset columns: ['labels', 'input_ids', 'attention_mask']
Validation dataset columns: ['labels', 'input_ids', 'attention_mask']


In [9]:
# ============================================================
# 9. Load NeoBERT sequence classifier and attach LoRA
# ============================================================

# Some NeoBERT remote-code revisions require xformers.ops.SwiGLU at import time.
# Instead of installing xformers and risking a PyTorch/CUDA conflict, this lightweight
# shim provides the SwiGLU class if xformers is not already installed.
install_xformers_swiglu_stub()

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

log(f"Loading NeoBERT sequence classifier on device={device}, dtype={torch_dtype}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    num_labels=2,
    problem_type="single_label_classification",
    torch_dtype=torch_dtype,
)

# NeoBERT remote code currently does not expose exactly the same forward signature
# expected by PEFT's PeftModelForSequenceClassification. In particular, PEFT may pass
# optional kwargs such as inputs_embeds/output_attentions/output_hidden_states, while
# NeoBERTForSequenceClassification.forward may not accept all of them. These optional
# fields are not needed for our input_ids-based LoRA classifier, so we safely drop any
# unsupported keyword arguments before the call reaches NeoBERT.
def patch_forward_to_ignore_unsupported_kwargs(module):
    original_forward = module.forward
    accepted_kwargs = set(inspect.signature(original_forward).parameters.keys())

    def wrapped_forward(*args, **kwargs):
        cleaned_kwargs = {k: v for k, v in kwargs.items() if k in accepted_kwargs}
        dropped_kwargs = sorted(set(kwargs.keys()) - set(cleaned_kwargs.keys()))
        # Print once for transparency, then stay quiet during training.
        if dropped_kwargs and not getattr(wrapped_forward, "_reported_dropped_kwargs", False):
            print("NeoBERT forward compatibility patch: dropping unsupported kwargs:", dropped_kwargs)
            wrapped_forward._reported_dropped_kwargs = True
        return original_forward(*args, **cleaned_kwargs)

    module.forward = wrapped_forward
    return module

model = patch_forward_to_ignore_unsupported_kwargs(model)

# Required for PEFT + gradient checkpointing in many Transformer workflows.
if hasattr(model, "enable_input_require_grads"):
    try:
        model.enable_input_require_grads()
    except Exception as exc:
        print("enable_input_require_grads skipped:", repr(exc))

if USE_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

# Attach LoRA to fused qkv attention projection.
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.to(device)

# Set pad token id consistently.
if getattr(model.config, "pad_token_id", None) is None:
    model.config.pad_token_id = tokenizer.pad_token_id


Installed lightweight xformers.ops.SwiGLU compatibility shim.
[16:46:41] Loading NeoBERT sequence classifier on device=cuda, dtype=torch.float16


config.json:   0%|          | 0.00/928 [00:00<?, ?B/s]

model.py:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

rotary.py:   0%|          | 0.00/2.58k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- model.py
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/981M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] NeoBERTForSequenceClassification LOAD REPORT from: chandar-lab/NeoBERT
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
dense.bias        | MISSING    | 
classifier.bias   | MISSING    | 
dense.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] NeoBERTForSequenceClassification does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


trainable params: 689,666 || all params: 222,947,332 || trainable%: 0.3093


In [10]:
# ============================================================
# 10. Class weights and Trainer definition
# ============================================================

train_labels = np.asarray(train_df[LABEL_COLUMN].values).astype(int)
n_pos = int((train_labels == 1).sum())
n_neg = int((train_labels == 0).sum())

if n_pos == 0 or n_neg == 0:
    raise RuntimeError("Training split has only one class; cannot compute class weights.")

# CrossEntropyLoss weight order: [class_0_weight, class_1_weight]
# This balanced formula makes minority vulnerable class more important.
total = n_pos + n_neg
class_weights = torch.tensor(
    [total / (2.0 * n_neg), total / (2.0 * n_pos)],
    dtype=torch.float32,
)

log(f"Train negatives={n_neg:,}, positives={n_pos:,}")
log(f"Class weights [safe, vulnerable] = {class_weights.tolist()}")

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # NeoBERT is loaded in float16 on GPU to save memory.
        # F.cross_entropy requires logits and class weights to have a compatible dtype.
        # Compute the loss in float32 for numerical stability, while gradients still
        # flow correctly back through the LoRA parameters.
        logits_for_loss = logits.float()
        labels_for_loss = labels.view(-1).long()
        weights = (
            self.class_weights.to(device=logits_for_loss.device, dtype=logits_for_loss.dtype)
            if self.class_weights is not None
            else None
        )
        loss = F.cross_entropy(logits_for_loss.view(-1, 2), labels_for_loss, weight=weights)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    y_true = labels.astype(int)
    y_score = softmax_positive(logits)
    return {
        "pr_auc": average_precision_score(y_true, y_score),
        "roc_auc": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else float("nan"),
        "f1_at_050": f1_score(y_true, (y_score >= 0.50).astype(int), zero_division=0),
        "precision_at_050": precision_score(y_true, (y_score >= 0.50).astype(int), zero_division=0),
        "recall_at_050": recall_score(y_true, (y_score >= 0.50).astype(int), zero_division=0),
        "mcc_at_050": matthews_corrcoef(y_true, (y_score >= 0.50).astype(int)) if len(np.unique((y_score >= 0.50).astype(int))) > 1 else 0.0,
    }

[16:46:56] Train negatives=384, positives=128
[16:46:56] Class weights [safe, vulnerable] = [0.6666666865348816, 2.0]


In [11]:
# ============================================================
# 11. TrainingArguments, collator, and Trainer
# ============================================================

if RUN_SMOKE_TEST:
    run_tag = "smoke"
elif RUN_PILOT_FOLD:
    run_tag = f"pilot_fold_{FOLD_TO_RUN}_train{PILOT_TRAIN_ROWS}_valid{PILOT_VALID_ROWS}"
else:
    run_tag = f"full_fold_{FOLD_TO_RUN}"
run_output_dir = CS2_OUTPUT_DIR / run_tag
run_output_dir.mkdir(parents=True, exist_ok=True)

# Estimate warmup_steps explicitly. Recent Transformers versions deprecate warmup_ratio.
effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
steps_per_epoch = math.ceil(len(train_ds) / max(1, effective_batch_size))
estimated_total_steps = int(steps_per_epoch * NUM_TRAIN_EPOCHS)
warmup_steps = max(1, int(WARMUP_RATIO * estimated_total_steps))

training_args_kwargs = dict(
    output_dir=str(run_output_dir / "trainer_checkpoints"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc",
    greater_is_better=True,
    # Do NOT enable Trainer fp16 here.
    #
    # NeoBERT is loaded with torch_dtype=float16 on CUDA to save memory. With PEFT,
    # some trainable adapter/classifier parameters can therefore produce FP16
    # gradients. Recent Accelerate/Trainer then fails during GradScaler unscale with:
    #     ValueError: Attempting to unscale FP16 gradients.
    #
    # We keep the model memory-saving dtype but disable Trainer's extra fp16
    # mixed-precision/GradScaler path. This is the safest Colab-compatible setting
    # for the current NeoBERT remote-code + PEFT stack.
    fp16=False,
    bf16=False,
    report_to=[],
    remove_unused_columns=False,
    # Use 0 workers for clearer Colab error traces and fewer tokenizer/collator surprises.
    dataloader_num_workers=0,
    label_names=["labels"],
    seed=SEED,
)

# Transformers changed evaluation_strategy -> eval_strategy in newer versions.
training_args_sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in training_args_sig.parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

if "gradient_checkpointing" in training_args_sig.parameters:
    training_args_kwargs["gradient_checkpointing"] = USE_GRADIENT_CHECKPOINTING

# Length bucketing reduces the probability that many long examples share a batch.
# It is useful even with batch size 1 because it makes accumulation/evaluation order more predictable.
if "group_by_length" in training_args_sig.parameters:
    training_args_kwargs["group_by_length"] = True
if "length_column_name" in training_args_sig.parameters:
    training_args_kwargs["length_column_name"] = "length"
if "torch_empty_cache_steps" in training_args_sig.parameters:
    training_args_kwargs["torch_empty_cache_steps"] = 10

training_args = TrainingArguments(**training_args_kwargs)

# Dynamic padding keeps batches compact and avoids padding every function to MAX_LENGTH.
# The wrapper below also defensively drops any non-model metadata columns if they
# accidentally remain in the Dataset.
base_data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

class ModelInputOnlyCollator:
    def __init__(self, base_collator):
        self.base_collator = base_collator
        self.allowed_keys = {"input_ids", "attention_mask", "token_type_ids", "labels"}

    def __call__(self, features):
        cleaned = [
            {k: v for k, v in feature.items() if k in self.allowed_keys}
            for feature in features
        ]
        return self.base_collator(cleaned)

data_collator = ModelInputOnlyCollator(base_data_collator)

# Transformers >= 5 removed/deprecated the Trainer(tokenizer=...) argument and uses
# processing_class instead. This adaptive block supports both old and new versions.
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer_sig = inspect.signature(Trainer.__init__)
if "processing_class" in trainer_sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig.parameters:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    print("Trainer.__init__ has neither processing_class nor tokenizer; continuing without either.")

trainer = WeightedLossTrainer(**trainer_kwargs)

print("TrainingArguments summary")
print(training_args)
print(f"Estimated total optimization steps: {estimated_total_steps}")
print(f"Warmup steps: {warmup_steps}")
print("Trainer fp16:", training_args.fp16)
print("Trainer bf16:", getattr(training_args, "bf16", None))
print("Trainer tokenizer/processor argument used:", "processing_class" if "processing_class" in trainer_kwargs else ("tokenizer" if "tokenizer" in trainer_kwargs else "none"))


TrainingArguments summary
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=In

In [12]:
# ============================================================
# 12. Train LoRA adapter
# ============================================================

# After a previous OOM, a runtime restart is usually safest. This cache clear helps
# for normal repeated cell execution, but cannot always recover a fragmented CUDA heap.
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

log("Starting NeoBERT LoRA training...")
start = time.time()
try:
    train_result = trainer.train()
except torch.cuda.OutOfMemoryError as exc:
    print("\nCUDA OOM during training.")
    print("Current MAX_LENGTH:", MAX_LENGTH)
    print("Current PER_DEVICE_TRAIN_BATCH_SIZE:", PER_DEVICE_TRAIN_BATCH_SIZE)
    print("Recommended recovery:")
    print("1. Runtime → Restart runtime")
    print("2. Keep MAX_LENGTH=1024, or reduce to 512 if this still fails")
    print("3. Keep PER_DEVICE_TRAIN_BATCH_SIZE=1")
    print("4. Do not attempt 4096 tokens on a 14–16 GB Colab GPU")
    raise
elapsed = time.time() - start

log(f"Training completed in {elapsed / 60:.2f} minutes.")

train_metrics = train_result.metrics
train_metrics["training_elapsed_seconds"] = elapsed
train_metrics["max_length"] = int(MAX_LENGTH)
train_metrics["per_device_train_batch_size"] = int(PER_DEVICE_TRAIN_BATCH_SIZE)
train_metrics["gradient_accumulation_steps"] = int(GRADIENT_ACCUMULATION_STEPS)
write_json(run_output_dir / "train_metrics.json", train_metrics)

# Save LoRA adapter and classification head modules.
adapter_dir = run_output_dir / "neobert_lora_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

log(f"Saved LoRA adapter to: {adapter_dir}")
if torch.cuda.is_available():
    print("Peak GPU memory GB:", round(torch.cuda.max_memory_allocated() / (1024**3), 3))


[16:46:57] Starting NeoBERT LoRA training...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


NeoBERT forward compatibility patch: dropping unsupported kwargs: ['inputs_embeds']


Epoch,Training Loss,Validation Loss


ValueError: Input contains NaN.

In [ ]:
# ============================================================
# 13. Validation prediction, threshold tuning, and artifact saving
# ============================================================

log("Predicting validation fold...")
pred_output = trainer.predict(valid_ds)
logits = pred_output.predictions
y_true = np.asarray(pred_output.label_ids).astype(int)
y_score = softmax_positive(logits)

ranking_metrics = {
    "n_samples": int(len(y_true)),
    "vulnerable_1": int((y_true == 1).sum()),
    "non_vulnerable_0": int((y_true == 0).sum()),
    "positive_rate": float((y_true == 1).mean()),
    "average_precision_pr_auc": float(average_precision_score(y_true, y_score)),
    "roc_auc": float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true)) == 2 else float("nan"),
    "fold": int(FOLD_TO_RUN),
    "run_smoke_test": bool(RUN_SMOKE_TEST),
    "run_pilot_fold": bool(RUN_PILOT_FOLD),
    "run_one_full_fold": bool(RUN_ONE_FULL_FOLD),
    "train_rows_used": int(len(train_df)),
    "valid_rows_used": int(len(valid_df)),
    "max_length": int(MAX_LENGTH),
    "model_name": MODEL_NAME,
    "code_column": CODE_COLUMN,
    "lora_target_modules": LORA_TARGET_MODULES,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
}

best_thr = select_best_f1_threshold(y_true, y_score)
operating_metrics_050 = compute_binary_metrics_from_scores(y_true, y_score, threshold=0.50)
operating_metrics_best = compute_binary_metrics_from_scores(y_true, y_score, threshold=best_thr["threshold"])
operating_metrics_best["threshold_strategy"] = "validation_f1_grid"
operating_metrics_best["selected_validation_f1"] = best_thr["f1"]

# Add metadata.
for d in [operating_metrics_050, operating_metrics_best]:
    d["fold"] = int(FOLD_TO_RUN)
    d["run_smoke_test"] = bool(RUN_SMOKE_TEST)
    d["run_pilot_fold"] = bool(RUN_PILOT_FOLD)
    d["run_one_full_fold"] = bool(RUN_ONE_FULL_FOLD)
    d["train_rows_used"] = int(len(train_df))
    d["valid_rows_used"] = int(len(valid_df))
    d["max_length"] = int(MAX_LENGTH)
    d["model_name"] = MODEL_NAME
    d["code_column"] = CODE_COLUMN

pred_df = pd.DataFrame({
    ID_COLUMN: valid_df[ID_COLUMN].values,
    PROJECT_COLUMN: valid_df[PROJECT_COLUMN].astype(str).values,
    "label": y_true,
    "y_score": y_score,
    "y_pred_050": (y_score >= 0.50).astype(int),
    "y_pred_selected_threshold": (y_score >= best_thr["threshold"]).astype(int),
    "selected_threshold": best_thr["threshold"],
    "fold": int(FOLD_TO_RUN),
})

pred_path = run_output_dir / "validation_predictions.parquet"
pred_df.to_parquet(pred_path, index=False)

write_json(run_output_dir / "ranking_metrics.json", ranking_metrics)
write_json(run_output_dir / "operating_metrics_threshold_050.json", operating_metrics_050)
write_json(run_output_dir / "operating_metrics_selected_threshold.json", operating_metrics_best)

print("Ranking metrics:")
print(json.dumps(ranking_metrics, indent=2))

print("\nOperating metrics at threshold 0.50:")
print(json.dumps(operating_metrics_050, indent=2))

print("\nOperating metrics at selected threshold:")
print(json.dumps(operating_metrics_best, indent=2))

log(f"Saved predictions to: {pred_path}")
log(f"Saved metrics to: {run_output_dir}")

In [ ]:
# ============================================================
# 14. Compact result table for the report
# ============================================================

summary_rows = [
    {
        "case_study": "CS2",
        "experiment": "NeoBERT-LoRA Phase 1",
        "scope": "smoke validation fold" if RUN_SMOKE_TEST else ("pilot project-disjoint development fold" if RUN_PILOT_FOLD else "one full project-disjoint development fold"),
        "representation": CODE_COLUMN,
        "fold": FOLD_TO_RUN,
        "max_length": MAX_LENGTH,
        "pr_auc": ranking_metrics["average_precision_pr_auc"],
        "roc_auc": ranking_metrics["roc_auc"],
        "selected_threshold": operating_metrics_best["threshold"],
        "precision": operating_metrics_best["precision"],
        "recall": operating_metrics_best["recall"],
        "f1": operating_metrics_best["f1"],
        "mcc": operating_metrics_best["mcc"],
        "predicted_positive_rate": operating_metrics_best["predicted_positive_rate"],
        "output_dir": str(run_output_dir),
    }
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(run_output_dir / "summary_table.csv", index=False)

## 15. How to move from smoke test to a reportable pilot fold

After the 1024-token smoke test succeeds, run the pilot fold first:

```python
RUN_SMOKE_TEST = False
RUN_PILOT_FOLD = True
RUN_ONE_FULL_FOLD = False
RUN_FULL_5FOLD_CV = False
PILOT_TRAIN_ROWS = 8192
PILOT_VALID_ROWS = 4096
```

The pilot fold is still not equivalent to a full 5-fold transformer CV, but it is much more meaningful than smoke because validation is sampled naturally from held-out projects.

Only after the pilot works and you have enough GPU time, try one full fold:

```python
RUN_SMOKE_TEST = False
RUN_PILOT_FOLD = False
RUN_ONE_FULL_FOLD = True
RUN_FULL_5FOLD_CV = False
```

Keep:

```python
RUN_FINAL_OUTER_HOLDOUT = False  # there is intentionally no holdout evaluation cell
```

Then rerun the notebook from the beginning.

## 16. How this connects to Phase 2 ReFT / HEFT

This notebook gives the Phase 1 LoRA baseline:

```text
NeoBERT + LoRA → binary vulnerability classifier
```

The next notebook should load the saved LoRA adapter and add a ReFT intervention on hidden states. That creates the HEFT-style pipeline:

```text
Phase 1: LoRA learns global syntax/security adaptation
Phase 2: ReFT learns small hidden-state interventions for vulnerability-specific cues
```

Do not implement ReFT before confirming that this LoRA notebook runs successfully on at least one project-disjoint fold.

## 17. Memory note for NeoBERT context length

The design slide mentions a 4096-token context. NeoBERT supports that context length, but training with 4096-token self-attention is usually too large for a 14–16 GB Colab GPU in the current remote-code SDPA path. Use `MAX_LENGTH=1024` for the reproducible Colab experiment. You can report this as a hardware-aware Phase-1 LoRA setting, while noting that 4096-token training is reserved for larger GPUs or additional memory optimizations.

Do not switch to `MAX_LENGTH=4096` on a T4/L4 unless the 1024-token run succeeds and you are prepared for OOM recovery.
